In [1]:
import requests
from datetime import datetime
import os
from dotenv import load_dotenv
import pandas as pd
import json
import re
import html
import numpy as np
load_dotenv()
api_key = os.getenv("API_KEY")

In [2]:
class ExtractChampionData:

    def __init__(self):
        self.last_version = self.get_latest_version()
        self.list_champ = self.get_all_champ_general_data()
        self.list_champ = list(self.list_champ.keys())

    def get_latest_version(self):
        """Get the last version of the game"""
        url = "https://ddragon.leagueoflegends.com/api/versions.json"
        rep = requests.get(url).json()
        latest = rep[0]
        return latest

    def get_all_champ_general_data(self):
        """Get the champion data"""
        url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/data/fr_FR/champion.json"
        resp = requests.get(url).json()
        return resp['data']
    
    def get_details_champ_data(self):
        """Get detail champion data"""
        data = []
        for champ in self.list_champ:
            url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/data/fr_FR/champion/{champ}.json"
            resp = requests.get(url).json()['data']
            data.append(resp[champ])
        return data

    # def download_all_png_champion(self):
    #     """Download all champions pictures"""
    #     for champ in self.list_champ:
    #         url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/img/champion/{champ}.png"
    #         path_save = fr"C:\Users\najim\Documents\Projets\LeagueOfLegends\images\{champ}.png"

    #         resp = requests.get(url)
    #         with open(path_save, "wb") as file:
    #             file.write(resp.content)

In [70]:
class TransformChampionData:

    def __init__(self):
        self.keys_to_drop = ['image', 'skins', 'blurb']
        self.table_champion = ['key', 'name', 'title', 'lore', 'tags', 'partype']
        self.table_champ_passive = ['key', 'passive']
        self.table_champ_info = ['key', 'info', 'allytips', 'enemytips']
        self.table_champ_spells = ['key', 'spells']
        self.table_champ_stats = ['key', 'stats']
    
    def drop_keys(self, data : list) -> list:
        new_data = []
        for dict_champ in data:
            dict_champ = {k: v for k,v in dict_champ.items() if k not in self.keys_to_drop}
            new_data.append(dict_champ)
        return new_data
    
    def dispatch_data(self, data : list):
        table_champion = []
        table_champ_passive = []
        table_champ_info = []
        table_champ_spells = []
        table_champ_stats = []
        
        for dict_champ in data:
            champion = {k: v for k,v in dict_champ.items() if k in self.table_champion}
            champ_passive = {k: v for k,v in dict_champ.items() if k in self.table_champ_passive}
            champ_info = {k: v for k,v in dict_champ.items() if k in self.table_champ_info}
            champ_spells = {k: v for k,v in dict_champ.items() if k in self.table_champ_spells}
            champ_stats = {k: v for k,v in dict_champ.items() if k in self.table_champ_stats}

            table_champion.append(champion)
            table_champ_passive.append(champ_passive)
            table_champ_info.append(champ_info)
            table_champ_spells.append(champ_spells)
            table_champ_stats.append(champ_stats)
            
        return table_champion, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats
    
    def split_stats(self, data : list):
        stats_cst, stats_up = [], []
        for stat in data:
            cst = {k:v for k,v in stat.items() if "perlevel" not in k}
            up = {k: v for k, v in stat.items() if "perlevel" in k or "key" in k}
            stats_cst.append(cst)
            stats_up.append(up)
        return stats_cst, stats_up

    def listing_into_text(self, data : list, key : str, sep : str = " "):
        new_data = []
        for champ in data:
            new_value = f"{sep}".join(champ[key])
            champ[key] = self._clean_text(new_value)
            new_data.append(champ)
        return new_data

    def dict_into_first_level(self, data : list, key_to_flat : str) -> list:
        new_data = []
        for champ in data:
            base = {k: v for k, v in champ.items() if k != key_to_flat}
            base.update(champ.get(key_to_flat, {}))
            new_data.append(base)
        return new_data
    
    def pop_key(self, data : list, key_to_remove : list[str]) -> list:
        new_data = []
        for i in data:
            for key in key_to_remove:
                d = i.pop(key, None)
            new_data.append(i)
        return new_data
    
    def _clean_text(self, desc : str) -> str:
        desc = re.sub(r"<[^>]+>|<[a-z]>", " ", desc)
        desc = re.sub(r"\xa0", " ", desc)
        desc = re.sub(r"_ClientTooltipWrapper|Wrapper", "", desc)
        desc = re.sub(r"\{\{.*?\}\}", "XX", desc)
        desc = re.sub(r"XX$", "", desc)
        desc = re.sub(r"[ ]+", " ", desc)
        return desc

    def text_cleaning(self, data : list, key : str) -> list:
        clean = []
        for champ in data:
            desc = self._clean_text(champ[key])
            champ[key] = desc
            clean.append(champ)
        return clean
    
    def clean_spells_data(self, data: list):
        new_data = []
        for champ in data:
            idchamp = champ['key']
            spells = champ['spells']

            spells = self.pop_key(
                spells,
                key_to_remove=['leveltip', 'description', 'cooldown', 'cost', 'datavalues',
                    'effect', 'vars', 'image', 'effectBurn', 'range', 'resource'])

            spells_clean = self.text_cleaning(spells, 'tooltip')
            spells_clean = self.text_cleaning(spells, 'name')
            spells_clean = self.text_cleaning(spells, 'id')

            for spell in spells_clean:
                spell['key'] = idchamp
                new_data.append(spell)

        return new_data
    
    def correct_spell_name(self, df_spells : pd.DataFrame, df_champ : pd.DataFrame) -> pd.DataFrame:
        """Rectifie les noms des spells de chaque champion"""

        df_spells = df_spells.rename(columns={'name':'spell_name'})
        df_merge = pd.merge(df_spells, df_champ[['key', 'name']], on='key')
        letters = ['Q', 'W', 'E', 'R']

        # numérotation 0,1,2,3 par groupe de 'name'
        df_merge['spell_rank'] = df_merge.groupby('name').cumcount()
        # on mappe 0→Q, 1→W, 2→E, 3→R
        df_merge['spell_id'] = df_merge['name'] + df_merge['spell_rank'].map(lambda x: letters[x])
        df_merge['spell_rank'] = df_merge['spell_rank'] + 1
        df_merge = df_merge.drop(columns=['name', 'id'])
        df_merge = df_merge.rename(columns={'key':'champ_id'})
        df_merge = df_merge[['champ_id', 'spell_id', 'spell_name', 'tooltip', 'maxrank', 'cooldownBurn', 'costBurn', 
                             'costType', 'maxammo', 'rangeBurn', 'spell_rank', 'version']]
        return df_merge
        
    def correct_spell_costype(self, df_spells : pd.DataFrame, df_champ : pd.DataFrame) -> pd.DataFrame:
        """Rectifie les resources consommées de chaque spells"""

        df_merge = pd.merge(df_spells, df_champ[['key', 'partype']], left_on='champ_id', right_on='key')

        df_merge['costType'] = df_merge['costType'].str.strip()

        df_merge['costType'] = np.where(
            df_merge['costType'] == r"{{ abilityresourcename }}", 
            df_merge['partype'], 
            df_merge['costType'])
        
        expression = r"\(?\{\{.*?\}\}\)?|\+|\."
        df_merge["costType"] = df_merge["costType"]\
            .str.replace(expression, "", regex=True)\
            .str.replace(r"\s+", ' ', regex=True)\
            .str.lower()

        return df_merge.drop(columns=['key', 'partype'])

    def transform_to_df(self, data : list, version : str) -> pd.DataFrame:
        """Transforme une liste de dictionnaire en dataframe pandas"""
        data = pd.DataFrame(data)
        data['version'] = version
        return data
        

In [71]:
def pipeline_champion():
    """Pipeline d'application des méthodes d'extraction, de transformation et de chargement"""
    
    extract = ExtractChampionData()
    lastest_version = extract.last_version
    print("Lastest version :", lastest_version)
    transform = TransformChampionData()

    details = extract.get_details_champ_data()
    details = transform.drop_keys(details)
    
    table_champion, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats = transform.dispatch_data(details)
    
    table_champion = transform.listing_into_text(table_champion, "tags", sep=' / ')
    table_champion = transform.text_cleaning(table_champion, key='lore')
    table_champion = transform.text_cleaning(table_champion, key='title')
    
    table_champ_info = transform.listing_into_text(table_champ_info, key="enemytips", sep=' ')
    table_champ_info = transform.listing_into_text(table_champ_info, key="allytips", sep=' ')
    table_champ_info = transform.dict_into_first_level(table_champ_info, key_to_flat='info')

    table_champ_passive = transform.dict_into_first_level(table_champ_passive, key_to_flat='passive')
    table_champ_passive = transform.pop_key(table_champ_passive, key_to_remove=['image'])
    table_champ_passive = transform.text_cleaning(table_champ_passive, key='name')
    table_champ_passive = transform.text_cleaning(table_champ_passive, key='description')

    table_champ_stats = transform.dict_into_first_level(table_champ_stats, key_to_flat='stats')
    table_champ_stats, table_champ_stats_up = transform.split_stats(table_champ_stats)

    table_champ_spells = transform.clean_spells_data(table_champ_spells)

    table_champion = transform.transform_to_df(table_champion, version=lastest_version)
    table_champ_info = transform.transform_to_df(table_champ_info, version=lastest_version)
    table_champ_passive = transform.transform_to_df(table_champ_passive, version=lastest_version)
    table_champ_stats = transform.transform_to_df(table_champ_stats, version=lastest_version)
    table_champ_stats_up = transform.transform_to_df(table_champ_stats_up, version=lastest_version)
    table_champ_spells = transform.transform_to_df(table_champ_spells, version=lastest_version)

    table_champ_spells = transform.correct_spell_name(table_champ_spells, table_champion)
    table_champ_spells = transform.correct_spell_costype(table_champ_spells, table_champion)
    
    return table_champion, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats, table_champ_stats_up

table_champion, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats, table_champ_stats_up = pipeline_champion()

Lastest version : 16.5.1


In [84]:
def get_max_length(col : pd.Series):
    x = 0
    for i in col.astype(str):
        if len(i) > x:
            x = len(i)
    return x

def create_schema(table : pd.DataFrame):
    schema = []
    for i in table.columns:
        name = i
        col_type = table[i].dtype.name
        max_length = get_max_length(table[i])
        schema.append({
            'colonne' : name,
            'type' : col_type,
            'longueur' : max_length
        })
    return pd.DataFrame(schema)

In [85]:
display(table_champion.head(5))
display(schema_info := create_schema(table_champion))

,key,name,title,lore,tags,partype,version
0,266,Aatrox,Épée des Darkin,"Autrefois, Aatrox et ses frères étaient honoré...",Fighter,Puits de sang,16.5.1
1,103,Ahri,Renarde à neuf queues,"Connectée à la magie du royaume spirituel, Ahr...",Mage / Assassin,Mana,16.5.1
2,84,Akali,Assassine rebelle,Ayant abandonné l'Ordre Kinkou et le titre de ...,Assassin,Énergie,16.5.1
3,166,Akshan,Sentinelle rebelle,"Se jouant du danger, Akshan combat le mal sans...",Marksman / Assassin,Mana,16.5.1
4,12,Alistar,Minotaure,Alistar est un guerrier redoutable cherchant à...,Tank / Support,Mana,16.5.1


,colonne,type,longueur
0,key,str,3
1,name,str,15
2,title,str,27
3,lore,str,705
4,tags,str,19
5,partype,str,14
6,version,str,6


In [86]:
display(table_champ_info.head(5))
display(schema_info := create_schema(table_champ_info))

,key,allytips,enemytips,attack,defense,magic,difficulty,version
0,266,Utilisez Ruée obscure tout en lançant Épée des...,Les attaques d'Aatrox sont prévisibles. Profit...,8,4,3,4,16.5.1
1,103,"Utilisez Charme pour préparer vos combos, cela...",Les capacités de survie d'Ahri sont considérab...,3,4,8,5,16.5.1
2,84,Akali peut facilement tuer les champions fragi...,"Quand Akali est occultée par Linceul nébuleux,...",5,3,8,7,16.5.1
3,166,"Se jouant du danger, Akshan combat le mal sans...",,0,0,0,0,16.5.1
4,12,Atomisation peut vous aider à mieux vous place...,"Alistar peut être dangereux, mais il est solid...",6,9,5,7,16.5.1


,colonne,type,longueur
0,key,str,3
1,allytips,str,800
2,enemytips,str,584
3,attack,int64,2
4,defense,int64,2
5,magic,int64,2
6,difficulty,int64,2
7,version,str,6


In [91]:
display(table_champ_passive.head(5))
display(schema_info := create_schema(table_champ_passive))

,key,name,description,version
0,266,Posture du massacreur,"Régulièrement, la prochaine attaque de base d'...",16.5.1
1,103,Vol essentiel,"Après avoir tué 9 sbires ou monstres, Ahri réc...",16.5.1
2,84,Marque d'assassin,Blesser un champion avec une compétence crée u...,16.5.1
3,166,Fourberie,Tous les trois coups venant de ses attaques ou...,16.5.1
4,12,Cri triomphant,Alistar charge son cri en étourdissant ou en d...,16.5.1


,colonne,type,longueur
0,key,str,3
1,name,str,31
2,description,str,520
3,version,str,6


In [88]:
display(table_champ_spells.head(5))
display(schema_info := create_schema(table_champ_spells))

,champ_id,spell_id,spell_name,tooltip,maxrank,cooldownBurn,costBurn,costType,maxammo,rangeBurn,spell_rank,version
0,266,AatroxQ,Épée des Darkin,"Aatrox abat son épée, infligeant XX pts de dég...",5,14/12/10/8/6,0,pas de coût,-1,25000,1,16.5.1
1,266,AatroxW,Chaînes infernales,"Aatrox lance une chaîne, ralentissant le premi...",5,20/18/16/14/12,0,pas de coût,-1,825,2,16.5.1
2,266,AatroxE,Ruée obscure,Passive : Aatrox récupère des PV équivalents ...,5,9/8/7/6/5,0,pas de coût,-1,25000,3,16.5.1
3,266,AatroxR,Fossoyeur des mondes,"Aatrox révèle sa vraie forme démoniaque, effra...",3,120/100/80,0,pas de coût,-1,25000,4,16.5.1
4,103,AhriQ,Orbe d'illusion,"Ahri lance son orbe et le ramène vers elle, in...",5,7,55/65/75/85/95,mana,-1,970,1,16.5.1


,colonne,type,longueur
0,champ_id,str,3
1,spell_id,str,16
2,spell_name,str,40
3,tooltip,str,982
4,maxrank,int64,1
5,cooldownBurn,str,22
6,costBurn,str,19
7,costType,str,39
8,maxammo,str,2
9,rangeBurn,str,24


In [89]:
display(table_champ_stats.head(5))
display(schema_info := create_schema(table_champ_stats))

,key,hp,mp,movespeed,armor,spellblock,attackrange,hpregen,mpregen,crit,attackdamage,attackspeed,version
0,266,650,0,345,38,32,175,3.00,0.0,0,60,0.651,16.5.1
1,103,590,418,330,21,30,550,2.50,8.0,0,53,0.668,16.5.1
2,84,600,200,345,23,37,125,9.00,50.0,0,62,0.625,16.5.1
3,166,610,350,330,26,30,500,3.75,8.2,0,52,0.638,16.5.1
4,12,685,350,330,40,32,125,8.50,8.5,0,62,0.625,16.5.1


,colonne,type,longueur
0,key,str,3
1,hp,int64,3
2,mp,int64,5
3,movespeed,int64,3
4,armor,int64,2
5,spellblock,int64,2
6,attackrange,int64,3
7,hpregen,float64,4
8,mpregen,float64,5
9,crit,int64,1


In [92]:
display(table_champ_stats_up.head(5))
display(schema_info := create_schema(table_champ_stats_up))

,key,hpperlevel,mpperlevel,armorperlevel,spellblockperlevel,hpregenperlevel,mpregenperlevel,critperlevel,attackdamageperlevel,attackspeedperlevel,version
0,266,114,0.0,4.8,2.05,0.50,0.0,0,0,2.500,16.5.1
1,103,104,25.0,4.2,1.30,0.60,0.8,0,0,2.200,16.5.1
2,84,119,0.0,4.7,2.05,0.90,0.0,0,0,3.200,16.5.1
3,166,107,40.0,4.7,1.30,0.65,0.7,0,0,4.000,16.5.1
4,12,120,40.0,4.7,2.05,0.85,0.8,0,0,2.125,16.5.1


,colonne,type,longueur
0,key,str,3
1,hpperlevel,int64,3
2,mpperlevel,float64,4
3,armorperlevel,float64,4
4,spellblockperlevel,float64,4
5,hpregenperlevel,float64,4
6,mpregenperlevel,float64,4
7,critperlevel,int64,1
8,attackdamageperlevel,int64,1
9,attackspeedperlevel,float64,5


In [93]:
table_champ_stats_up['attackdamageperlevel'].describe()

count    172.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: attackdamageperlevel, dtype: float64